In [1]:
# Cell 1
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import xgboost as xgb

df = pd.read_csv('../data/highk_with_dielectric.csv')
df_filtered = df[(df['e_total'] > 1) & (df['e_total'] < 100)].copy()

model_k = xgb.XGBRegressor()
model_k.load_model('../data/xgb_dielectric_model.json')
feature_cols = model_k.get_booster().feature_names

print(f"데이터: {len(df_filtered)}개  features: {len(feature_cols)}개")

데이터: 619개  features: 132개


In [2]:
# Cell 2
importance = pd.Series(model_k.feature_importances_, index=feature_cols)
top20 = importance.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
top20[::-1].plot(kind='barh', color='darkorange')
plt.xlabel('Feature Importance (gain)')
plt.title('Top 20 Features — k (Dielectric Constant) Model')
plt.tight_layout()
plt.savefig('../data/feature_importance_k.png', dpi=150)
plt.show()

print('\nTop 10:')
print(top20.head(10).round(4))


Top 10:
MagpieData avg_dev NdUnfilled      0.1175
MagpieData range NdUnfilled        0.0859
MagpieData minimum NpValence       0.0844
MagpieData mean NdUnfilled         0.0778
MagpieData minimum AtomicWeight    0.0527
MagpieData avg_dev Row             0.0408
MagpieData maximum NdUnfilled      0.0263
MagpieData minimum NUnfilled       0.0248
MagpieData mean Column             0.0226
MagpieData minimum NpUnfilled      0.0205
dtype: float32


<Figure size 1000x700 with 1 Axes>

---
## 해석: 이온 분극이 유전율을 지배한다

### 유전율의 물리
전체 유전율 = 전자 분극 기여(ε_electronic) + 이온 분극 기여(ε_ionic)

**Clausius-Mossotti 관계:**
(ε-1)/(ε+2) = nα/3ε₀
- α: 분극률 (원자가 클수록, 전자가 많을수록 큼)
- n: 단위셀 당 원자 수

**k가 높은 소재의 특징:**
- Ba²⁺, Sr²⁺, Pb²⁺: 대형 연성 양이온 → 높은 이온 분극률 → k ↑
- Ti⁴⁺, Nb⁵⁺: 비어있는 d 오비탈 + 변위 가능한 구조 → 강유전 → k >> 100 (필터됨)
- Hf⁴⁺, Zr⁴⁺: 중간 크기 d⁰ 양이온 → k ≈ 20~25 (DRAM 최적 범위)
- Al³⁺: 소형 3+ 양이온 → 낮은 분극률 → k ≈ 9

### 상위 feature 해석 (실제 출력 기반)
| 순위 | Feature | 물리적 의미 |
|------|---------|------------|
| 1위 | avg_dev NdUnfilled | d 오비탈 빈자리의 원소간 편차 → 이온 분극 다양성 지표 |
| 2위 | range NdUnfilled | d 오비탈 빈자리 범위 → 전이금속 vs 비전이금속 조합 여부 |
| 3위 | minimum NpValence | p 가전자 최솟값 → 음이온(O²⁻ 등) 전자 구성 |

→ NdUnfilled(비어있는 d 오비탈 수)가 핵심: d⁰ 전이금속(Hf, Zr, Ti)은 NdUnfilled가 크고, 이들이 high-k 소재의 주요 양이온

### Phase 1 (밴드갭) vs Phase 2 (유전율) 핵심 비교
| | 밴드갭 모델 | 유전율 모델 |
|--|--|--|
| 1위 feature | avg_dev NdValence | avg_dev NdUnfilled |
| 지배 물리 | d 오비탈 점유 상태 | 이온 분극률 |
| 이론 기반 | 결정장 이론 | Clausius-Mossotti |
| 높은 값 조건 | d⁰ (빈 d 오비탈) | 대형·연성 양이온 |

→ 면접 메시지: "같은 132개 Magpie feature로 학습했는데 서로 완전히 다른 물리를 포착한다."

NdValence(d 전자 수) vs NdUnfilled(빈 d 오비탈 수): 같은 d 오비탈이지만 밴드갭은 '채워진 것', 유전율은 '비어있는 것'을 중시 — 물리적으로 정합성 있음.

In [3]:
# Cell 3
top_feature = top20.index[0]
second_feature = top20.index[1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df_filtered[top_feature], df_filtered['e_total'],
                alpha=0.3, s=10, color='darkorange')
axes[0].set_xlabel(top_feature)
axes[0].set_ylabel('e_total (dielectric constant)')
axes[0].set_title(f'Top Feature vs Dielectric Constant')

r1 = df_filtered[[top_feature, 'e_total']].corr().iloc[0, 1]
print(f'1위 feature ({top_feature}) vs e_total: r = {r1:.3f}')

axes[1].scatter(df_filtered[second_feature], df_filtered['e_total'],
                alpha=0.3, s=10, color='seagreen')
axes[1].set_xlabel(second_feature)
axes[1].set_ylabel('e_total')
axes[1].set_title(f'2위 Feature vs Dielectric Constant')

r2 = df_filtered[[second_feature, 'e_total']].corr().iloc[0, 1]
print(f'2위 feature ({second_feature}) vs e_total: r = {r2:.3f}')

plt.tight_layout()
plt.savefig('../data/interpretation_k_features.png', dpi=150)
plt.show()

1위 feature (MagpieData avg_dev NdUnfilled) vs e_total: r = 0.470
2위 feature (MagpieData range NdUnfilled) vs e_total: r = 0.437


<Figure size 1200x500 with 2 Axes>

In [4]:
# Cell 4
model_eg = xgb.XGBRegressor()
model_eg.load_model('../data/xgb_bandgap_model.json')
eg_features = model_eg.get_booster().feature_names
importance_eg = pd.Series(model_eg.feature_importances_, index=eg_features)

print('=== Phase 1 (밴드갭) Top 5 ===')
print(importance_eg.sort_values(ascending=False).head(5).round(4))
print()
print('=== Phase 2 (유전율) Top 5 ===')
print(importance.sort_values(ascending=False).head(5).round(4))
print()
print('→ 두 모델이 서로 다른 feature를 중심으로 작동함을 확인')

=== Phase 1 (밴드갭) Top 5 ===
MagpieData avg_dev NdValence    0.2454
MagpieData maximum NdValence    0.1039
MagpieData range NdValence      0.0581
MagpieData mean NdValence       0.0332
MagpieData avg_dev GSbandgap    0.0229
dtype: float32

=== Phase 2 (유전율) Top 5 ===
MagpieData avg_dev NdUnfilled      0.1175
MagpieData range NdUnfilled        0.0859
MagpieData minimum NpValence       0.0844
MagpieData mean NdUnfilled         0.0778
MagpieData minimum AtomicWeight    0.0527
dtype: float32

→ 두 모델이 서로 다른 feature를 중심으로 작동함을 확인
